# Kestrel-22M-A4M: дообучение на длинном контексте, оптимизаторы

Претрейн мы уже провели: два чекпоинта (A — AdamW, M — Muon) выложены на Hugging Face. Ноутбук скачивает их,
проверяет, что они корректно загружаются в **вашу** модель, считает статистику чекпоинтов и zero-shot оценку на длине 2048,
затем делает дообучение 2×2 (чем предобучали × чем дообучаем) с YaRN, строит графики и пишет `results.json`.
Вся логика лежит в пакете `kestrel`; здесь только запуск. Каждая обучающая секция сама пропускается, если её
итоговый чекпоинт уже есть на Drive, и продолжает с последнего промежуточного — после обрыва сессии просто
запустите ноутбук заново. Ожидаемое время: 40–60 минут на GPU.

## 1. Setup

GPU, Google Drive (все артефакты хранятся в `KESTREL_DIR`) и ваш проект.
Заархивируйте проект на своей машине в `kestrel-hw.zip` (без виртуального окружения и скачанных данных)
и положите архив на Google Drive: `MyDrive/kestrel_hw/kestrel-hw.zip` — так он переживёт обрыв сессии. Если архива нигде
нет, ячейка откроет окно загрузки файла. Архив распаковывается заново при каждом запуске, поэтому после правок кода
достаточно заменить архив и перезапустить ячейку.

In [ ]:
SMOKE = False  # True: быстрый прогон на CPU (tiny-модель, синтетические данные) — только чтобы проверить, что всё работает

import json, os, shutil, subprocess, sys, tempfile, zipfile
from pathlib import Path
import torch

ZIP_NAME = "kestrel-hw.zip"   # имя архива с вашим проектом

if SMOKE:
    DEVICE = "cpu"
    REPO = os.environ["KESTREL_REPO"]                                   # локальная копия проекта
    KESTREL_DIR = os.environ.get("KESTREL_DIR") or tempfile.mkdtemp(prefix="kestrel_smoke_")
    sys.path.insert(0, REPO)
else:
    assert torch.cuda.is_available(), "Включите GPU: Runtime → Change runtime type → T4 GPU"
    DEVICE = "cuda"
    gpu = torch.cuda.get_device_name(0)
    print("GPU:", gpu)
    if "T4" not in gpu:
        print("ВНИМАНИЕ: ожидается T4; время обучения и числа могут отличаться")
    from google.colab import drive
    drive.mount("/content/drive")
    KESTREL_DIR = "/content/drive/MyDrive/kestrel_hw/"
    REPO = "/content/kestrel-hw"
    # архив ищем на Диске (переживает обрыв сессии) и в /content; если его нет -- откроется окно загрузки
    zip_path = next((p for p in (KESTREL_DIR + ZIP_NAME, "/content/" + ZIP_NAME) if os.path.exists(p)), None)
    if zip_path is None:
        from google.colab import files
        uploaded = files.upload()
        zip_path = "/content/" + next(iter(uploaded))
    shutil.rmtree(REPO, ignore_errors=True)
    shutil.rmtree("/content/_unz", ignore_errors=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall("/content/_unz")
    # корень проекта -- папка с configs/kestrel.yaml (архив может быть с внешней папкой или без неё)
    root = next(p.parent.parent for p in Path("/content/_unz").rglob("kestrel.yaml")
                if p.parent.name == "configs" and "__MACOSX" not in p.parts)
    shutil.move(str(root), REPO)
    print("проект распакован из", zip_path)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO], check=True)
    sys.path.insert(0, REPO)

os.makedirs(KESTREL_DIR, exist_ok=True)
os.chdir(REPO)
print("REPO:", REPO, "| KESTREL_DIR:", KESTREL_DIR, "| device:", DEVICE)

## 2. Тесты

Публичные тесты должны проходить до начала обучения. Если что-то падает — исправьте реализацию и перезапустите ячейку.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-m", "not slow", "-p", "no:cacheprovider"],
                   cwd=REPO, capture_output=True, text=True)
print("\n".join(r.stdout.strip().splitlines()[-15:]))
if r.returncode != 0:
    raise SystemExit("Публичные тесты не проходят. Запустите `pytest -m 'not slow' -x` локально, исправьте "
                     "реализацию в пяти файлах с TODO и повторите.")

## 3. Конфиг, данные, чекпоинты

Хеш конфига проверяется (менять `configs/kestrel.yaml` нельзя). Данные (без обучающего сплита претрейна, он вам не
нужен) и два чекпоинта претрейна скачиваются с HuggingFace Hub в `KESTREL_DIR`. Если преподаватель ещё не опубликовал
чекпоинты, ячейка остановится с понятным сообщением.

In [ ]:
from kestrel.config import compute_config_hash, load_config, smoke_config, verify_config_hash
from kestrel.data.download import download_checkpoints, download_data
from kestrel.data.synthetic import make_synthetic_dataset
from kestrel.model import build_model, count_parameters
from kestrel.model.model import load_checkpoint

DATA_DIR = os.path.join(KESTREL_DIR, "data")
RUNS = os.path.join(KESTREL_DIR, "runs") + os.sep
FIGS = os.path.join(KESTREL_DIR, "figures") + os.sep
os.makedirs(RUNS, exist_ok=True); os.makedirs(FIGS, exist_ok=True)

if SMOKE:
    cfg = smoke_config()
    make_synthetic_dataset(DATA_DIR, cfg)
    CKPT_DIR = os.environ["KESTREL_CKPT_DIR"]                 # чекпоинты, которые сделал smoke-прогон преподавательского ноутбука
    REFERENCE = json.load(open(os.path.join(CKPT_DIR, "reference.json"), encoding="utf-8"))["val_loss"]
else:
    verify_config_hash()
    cfg = load_config()
    CKPT_DIR = os.path.join(KESTREL_DIR, "checkpoints")
    if not os.path.exists(os.path.join(DATA_DIR, "meta.json")):
        download_data(cfg, DATA_DIR, role="student")
    if not os.path.exists(os.path.join(CKPT_DIR, cfg.checkpoints.files["A"])):
        download_checkpoints(cfg, CKPT_DIR)
    REFERENCE = cfg.checkpoints.reference_val_loss

CK = {tag: os.path.join(CKPT_DIR, cfg.checkpoints.files[tag]) for tag in ("A", "M")}
PRETRAIN_LOGS = json.load(open(os.path.join(CKPT_DIR, cfg.checkpoints.files["logs"]), encoding="utf-8"))
print("параметров в модели:", f"{count_parameters(build_model(cfg, 'meta')):,}", "| чекпоинты:", CKPT_DIR)

## 4. Проверка чекпоинтов

Оба чекпоинта загружаются в вашу модель (`strict=True`), и на `pretrain_val` при длине 512 считается val loss. Он
должен совпасть со значением преподавателя с допуском `checkpoints.val_loss_tolerance`. Если не совпал, в вашей
реализации внимания, MoE или YaRN осталась ошибка, которую не поймали тесты (см. FAQ, приложение A.7).

In [ ]:
from kestrel.train.evaluate import checkpoint_val_loss, loss_by_position_buckets, passkey_accuracy, eval_loss
from kestrel.train.metrics import env_info, expert_load, matrix_report, student_code_hash, tests_hash
from kestrel.train.trainer import is_done, read_log, train

CHECK_BS = cfg.pretrain.micro_batch_size or cfg.pretrain.batch_size


def load(path, yarn_scale):
    model = build_model(cfg, DEVICE)
    load_checkpoint(model, path)
    model.set_yarn_scale(yarn_scale)
    return model


try:
    reference = {tag: float(REFERENCE[tag]) for tag in ("A", "M")}
except (TypeError, ValueError):
    raise SystemExit("Эталонные val loss чекпоинтов ещё не заполнены в configs/kestrel.yaml (checkpoints.reference_val_loss).")

checkpoint_check = {}
for tag in ("A", "M"):
    model = load(CK[tag], cfg.pretrain.yarn_scale)
    val = checkpoint_val_loss(model, DATA_DIR, cfg.pretrain.seq_len, CHECK_BS)
    ok = abs(val - reference[tag]) <= cfg.checkpoints.val_loss_tolerance
    checkpoint_check[tag] = {"val_loss": val, "reference": reference[tag], "ok": ok}
    print(f"{tag}: val loss {val:.4f} | у преподавателя {reference[tag]:.4f} | {'OK' if ok else 'НЕ СОВПАЛО'}")
if not all(c["ok"] for c in checkpoint_check.values()):
    raise SystemExit("Val loss выданных чекпоинтов не совпал с эталонным: в вашем коде есть ошибка, которую не поймали "
                     "тесты. Проверьте m² в scale, порядок нормы и RoPE, softmax в route (приложение A.7).")

## 5. Статистика чекпоинтов

Средний stable rank матриц группы `matrix`, нагрузка экспертов по слоям (по `pretrain_val`), рисунки 1 и 5.
Кривые претрейна берутся из логов преподавателя (`pretrain_logs.json`).

In [ ]:
import matplotlib.pyplot as plt
from kestrel.viz import plots

PRETRAIN_VAL = os.path.join(DATA_DIR, "pretrain_val.bin")
checkpoint_stats = {}
for tag in ("A", "M"):
    model = load(CK[tag], cfg.pretrain.yarn_scale)
    checkpoint_stats[tag] = {
        "mean_stable_rank": matrix_report(model)["mean_stable_rank"],
        "expert_load": expert_load(model, PRETRAIN_VAL, cfg.pretrain.seq_len, n_batches=8, batch_size=CHECK_BS),
    }
    print(tag, "stable rank:", round(checkpoint_stats[tag]["mean_stable_rank"], 3),
          "| макс. доля эксперта:", round(max(max(layer) for layer in checkpoint_stats[tag]["expert_load"]), 3))

pre_logs = {"AdamW": PRETRAIN_LOGS["A"], "Muon": PRETRAIN_LOGS["M"]}
plots.plot_loss_curves(pre_logs, "val_loss", "Претрейн: validation loss", FIGS + "fig1_pretrain_val.png")
plots.plot_loss_curves(pre_logs, "train_loss", "Претрейн: train loss", FIGS + "fig1_pretrain_train.png")
for tag in ("A", "M"):
    plots.plot_expert_load(checkpoint_stats[tag]["expert_load"], f"Нагрузка экспертов, чекпоинт {tag}",
                           FIGS + f"fig5_expert_load_{tag}.png")
plt.show()

## 6. Zero-shot на длинном контексте

Претрейн шёл на коротких последовательностях (`pretrain.seq_len`). Смотрим лосс по позициям внутри последовательности в
4 раза длиннее (`finetune.seq_len`) без дообучения: сначала без YaRN, затем с YaRN (scale = `finetune.yarn_scale`).

In [ ]:
LONG_SEQ = cfg.finetune.seq_len
BUCKET = LONG_SEQ // 4
LONG_VAL = os.path.join(DATA_DIR, "long_val.bin")
PASSKEY = os.path.join(DATA_DIR, "passkey_eval.jsonl")

zero_shot = {}
for tag in ("A", "M"):
    for yarn_tag, scale in (("no_yarn", 1.0), ("yarn", cfg.finetune.yarn_scale)):
        model = load(CK[tag], scale)
        zero_shot[f"{tag}/{yarn_tag}"] = {
            "buckets": loss_by_position_buckets(model, LONG_VAL, LONG_SEQ, BUCKET),
            "passkey": passkey_accuracy(model, PASSKEY, cfg.seeds.passkey_eval),
        }
        print(tag, yarn_tag, {k: round(v, 3) for k, v in zero_shot[f"{tag}/{yarn_tag}"]["buckets"].items()})

## 7. Дообучение на длинном контексте: схема 2×2

Четыре прогона: `A→A`, `A→M`, `M→A`, `M→M` (первая буква — чем предобучали, вторая — чем дообучаем).
YaRN scale = `finetune.yarn_scale`, состояние оптимизатора каждый раз новое, батч набирается из микробатчей.
Первые 50 шагов логируются на каждом шаге. После пятнадцати шагов ноутбук замеряет скорость и предупреждает,
если шаг подозрительно медленный (обычно это значит, что внимание считается не через `F.scaled_dot_product_attention`).

In [ ]:
FT, FT_RUN = {}, {}
NUM_WORKERS = 0 if SMOKE else 2
for pre in ("A", "M"):
    for ft_kind, ft_tag in (("adamw", "A"), ("muon", "M")):
        name = f"{pre}->{ft_tag}"
        run = f"ft_{pre}_{ft_tag}"
        res = train(build_model(cfg, DEVICE), ft_kind, "finetune", cfg, DATA_DIR, f"{RUNS}{run}.pt",
                    init_from=CK[pre], log_path=f"{RUNS}{run}.jsonl", num_workers=NUM_WORKERS)
        FT[name], FT_RUN[name] = res["final_path"], run
        print(f"{name}: val loss {res['val_loss']:.4f} | шаг {res['step_sec']:.2f} с")

## 8. Метрики

Финальный val loss, лосс по позициям, passkey, а также `matrix_report`: stable rank и дрейф матриц от весов
исходного чекпоинта.

In [ ]:
pre_state = {tag: torch.load(CK[tag], map_location="cpu", weights_only=True)["model"] for tag in ("A", "M")}


def max_first_steps(run, n=50):
    return max(r["train_loss"] for r in read_log(f"{RUNS}{run}.jsonl") if "train_loss" in r and r["step"] <= n)


def step_seconds(run):
    return next(r["step_sec"] for r in read_log(f"{RUNS}{run}.jsonl") if "step_sec" in r)


finetune_metrics, finetune_step_sec = {}, {}
for name, path in FT.items():
    model = load(path, cfg.finetune.yarn_scale)
    rep = matrix_report(model, ref_state=pre_state[name[0]])
    finetune_metrics[name] = {
        "val_loss": eval_loss(model, LONG_VAL, LONG_SEQ, None, batch_size=cfg.finetune.micro_batch_size or 2),
        "buckets": loss_by_position_buckets(model, LONG_VAL, LONG_SEQ, BUCKET),
        "passkey": passkey_accuracy(model, PASSKEY, cfg.seeds.passkey_eval),
        "mean_stable_rank": rep["mean_stable_rank"], "mean_drift_from_pretrain": rep["mean_drift"],
        "stable_rank_change": rep["mean_stable_rank"] - checkpoint_stats[name[0]]["mean_stable_rank"],
        "max_train_loss_first50": max_first_steps(FT_RUN[name]),
        "matrix_report": rep["per_matrix"],
    }
    finetune_step_sec[name] = step_seconds(FT_RUN[name])
    m = finetune_metrics[name]
    print(f"{name}: val {m['val_loss']:.4f} | passkey {m['passkey']['all']:.2f} | "
          f"stable rank {m['mean_stable_rank']:.2f} | drift {m['mean_drift_from_pretrain']:.4f}")

## 9. Графики

Сохраняются в `KESTREL_DIR/figures/`. Рисунки 1 и 5 уже построены в секции 5.

In [ ]:
ft_logs = {k: read_log(f"{RUNS}{run}.jsonl") for k, run in FT_RUN.items()}

plots.plot_loss_curves(ft_logs, "val_loss", "Дообучение: validation loss (длинный контекст)", FIGS + "finetune_val.png")
plots.plot_finetune_first_steps(ft_logs, 50, FIGS + "fig2_finetune_first50.png")
plots.plot_position_buckets({**{f"zero-shot {k}": v["buckets"] for k, v in zero_shot.items()},
                             **{f"после {k}": v["buckets"] for k, v in finetune_metrics.items()}},
                            out_path=FIGS + "fig3_position_buckets.png")
block = min(3, cfg.model.n_layers - 1)  # блок 4 (нумерация с 1)
name = f"layers.{block}.moe.experts.0.w_down.weight"
spectra = {f"{k} (до дообучения)": pre_state[k][name] for k in pre_state}
spectra.update({f"{k} (после)": torch.load(p, map_location="cpu", weights_only=True)["model"][name] for k, p in FT.items()})
plots.plot_singular_values(spectra, f"Сингулярные числа {name}", FIGS + "fig4_singular_values.png")
plt.show()

## 10. Таблицы для отчёта и results.json

Ячейка печатает таблицы 10–12 отчёта в готовом виде (числа с тремя знаками) — их можно копировать в `TECH_REPORT.md`.
Все числа, хеш вашего кода (SHA-256 по пяти файлам с TODO), хеши конфига и тестов, версия torch и название GPU
записываются в `results.json`. Этот файл сдаётся вместе с архивом проекта и заполненным `TECH_REPORT.md`.

In [ ]:
last = lambda buckets: list(buckets.values())[-1]
f3 = lambda x: f"{x:.3f}"

rows10 = [[f"{k} ({'AdamW' if k == 'A' else 'Muon'})", f3(checkpoint_check[k]["val_loss"]),
           f3(checkpoint_stats[k]["mean_stable_rank"]), f3(max(max(l) for l in checkpoint_stats[k]["expert_load"]))]
          for k in ("A", "M")]
rows11 = [[f"{k[0]}, {'YaRN s = ' + str(cfg.finetune.yarn_scale) if k.endswith('/yarn') else 'обычный RoPE'}"]
          + [f3(x) for x in v["buckets"].values()] for k, v in zero_shot.items()]
rows12 = [[k, f3(m["val_loss"]), f3(last(m["buckets"])), f"{100 * m['passkey']['all']:.1f}", f3(m["max_train_loss_first50"]),
           f3(m["mean_drift_from_pretrain"]), f3(m["stable_rank_change"])] for k, m in finetune_metrics.items()]
heads = {
    "Таблица 10": ["Модель", "Val loss на %d" % cfg.pretrain.seq_len, "Средний stable rank", "Макс. доля у эксперта"],
    "Таблица 11": ["Модель и режим"] + list(next(iter(zero_shot.values()))["buckets"]),
    "Таблица 12": ["Прогон", "Val loss на %d" % LONG_SEQ, "Лосс в последнем бакете", "Passkey, %",
                   "Макс. лосс, шаги 1-50", "Среднее ‖W−W₀‖/‖W₀‖", "Δ stable rank"],
}
report_tables = {}
for title, rows in (("Таблица 10", rows10), ("Таблица 11", rows11), ("Таблица 12", rows12)):
    report_tables[title] = {"columns": heads[title], "rows": rows}
    print(f"\n{title}")
    print("| " + " | ".join(heads[title]) + " |")
    print("|" + " --- |" * len(heads[title]))
    for r in rows:
        print("| " + " | ".join(r) + " |")

results = {
    "smoke": SMOKE,
    "student_code_sha256": student_code_hash(REPO),
    "config_sha256": None if SMOKE else compute_config_hash(),
    "tests_sha256": tests_hash(REPO),
    "env": env_info(),
    "n_parameters": count_parameters(build_model(cfg, "meta")),
    "checkpoint_check": checkpoint_check,
    "checkpoint_stats": checkpoint_stats,
    "zero_shot": zero_shot,
    "finetune": finetune_metrics,
    "finetune_step_sec": finetune_step_sec,
    "report_tables": report_tables,
    "logs": {"finetune": {k: [r for r in v if "val_loss" in r] for k, v in ft_logs.items()}},
}
path = os.path.join(KESTREL_DIR, "results.json")
with open(path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("\nзаписано:", path)